In [ ]:
%run WIRE_ISSUES_MODULE.py


In [ ]:
feature_names=['vehicle_id','device_model_id','sales_dept','operator_id','network','installation_subtype','alert_count','avg_duration','sum_of_duration','median_of_duration','intentional_wire_cut', 'maintainence_wire_cut','wire_loose', 'charge_to_ign', 'ign_wire_cut', 'ign_to_charge','driving_count','stoppage_count','time_slot','lat','long','displacement_travelled','current_state','displacements_mean','displacements_median','displacements_maximum','displacements_minimum']    


In [ ]:
len(feature_names
   )

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import matplotlib.pyplot as plt
%run WIRE_ISSUES_MODULE.py
import warnings
warnings.filterwarnings("ignore")





db = client.get_database('telematics_db')


db2 = client2.get_database('telematics_db')







In [ ]:
def loc_before_no_info(vid,time):
    time=int(time)
    url=GPS_API_BASE+"/gps/top?vehicleIds={}&fromTime={}&sortOrder=desc&timeCondition=lte".format(vid,time)
    #print(url)
    data=json.load(urllib.request.urlopen(url))
        
    try:
        loc=data['data'][str(vid)]['location']
            
    except:
        print('no_DATA')
        loc=None
        
    return loc
 


In [ ]:
def loc_after_no_info(vid,time):
    time=int(time)
    url=GPS_API_BASE+"/gps/top?vehicleIds={}&fromTime={}&sortOrder=asc&timeCondition=gte".format(vid,time)
    data=json.load(urllib.request.urlopen(url))
        
    try:
        loc=data['data'][str(vid)]['location']
            
    except:
        print('no_DATA')
        loc=None
        
    return loc

In [ ]:
def haversine(L1,L2):
    try:
        
        """
        Calculate the great circle distance between two points
        on the earth (specified in decimal degrees)
        """

        lon1=L1[0]
        lat1=L1[1]
        lon2=L2[0]
        lat2=L2[1]
        # convert decimal degrees to radians
        lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

        # haversine formula
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * asin(sqrt(a))
        r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
        return c * r
    except:
        return None
        


In [ ]:
def displacement(vid,fromTime,toTime):
    start_loc=loc_before_no_info(vid,fromTime)
    end_loc=loc_after_no_info(vid,toTime)
    displacement=haversine(start_loc,end_loc)
    return displacement
    
    

In [ ]:
def displacement_history(vid):
    try:
        
        device_history=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':tm.time()-30*86400},'toTime':{'$gte':0},'vehicleId':vid})))  
        if len(device_history)>0:
            displacements=[]
            for i , row in device_history.iterrows():
                    start_loc=loc_before_no_info(row['vehicleId'],row['fromTime'])
                    end_loc=loc_after_no_info(row['vehicleId'],row['toTime'])
                    displacement_value=haversine(start_loc,end_loc)
                    displacements.append(displacement_value)

            displacements=np.array(displacements)
            mean=displacements.mean()
            median=np.median(displacements)
            maximum=displacements.max()
            minimum=displacements.min()
        return [mean,median,maximum,minimum]
    
    except Exception as e:
        print(e)
        return [None,None,None,None]
        

In [ ]:

def no_info_duration_bucket(duration_sec):
    i=duration_sec
    if (i<1*86400):

        bucket="less than one day"   
    elif (i>=1*86400)&(i<3*86400):
        bucket="1-3 days"
    elif(i>=3*86400)&(i<7*86400):
        bucket="3-7 days"
    elif(i>=7*86400)&(i<15*86400):
        bucket="7-15 days"
    elif(i>=15*86400)&(i<30*86400):
        bucket="15-30 days"
    elif (i>=30*86400):
        bucket="30+ days"
    return bucket



In [ ]:
def time_slot(x):
    seconds=x%86400
    if seconds<=14400:
        slot=1
    elif (seconds>14400)&(seconds<14400*2):
        slot=2
    elif (seconds>14400*2)&(seconds<14400*3):
        slot=3
    elif (seconds>14400*3)&(seconds<14400*4):
        slot=4
    elif (seconds>14400*4)&(seconds<14400*5):
        slot=5
    elif (seconds>14400*5)&(seconds<14400*6):
        slot=6
    return slot
   

In [ ]:
def last_state_itenarery(vid,fromTime):
    last_iteniraray=list(db2.itinerary.find({'vehicleId':vid,'fromTime':{'$lte':fromTime}}).sort('fromTime',-1).limit(1))[0]        
    last_state=last_iteniraray['mode']
    return last_state

In [ ]:
def next_state_itenarery(vid,fromTime):
    last_iteniraray=list(db2.itinerary.find({'vehicleId':vid,'fromTime':{'$gte':fromTime}}).sort('fromTime',1).limit(1))[0]        
    last_state=last_iteniraray['mode']
    return last_state

In [ ]:
def last_state_before_no_info_history(vid):
    """returns number of driving state and stoppage state before the alerts"""
    device_history=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':tm.time()-30*86400},'toTime':{'$gte':0},'vehicleId':vid})))
    driving=0
    stoppage=0
    if len(device_history)>0:
        for i,row in device_history.iterrows():
            if last_state_itenarery(vid,row['fromTime'])=='DRIVING':
                driving+=1
            elif last_state_itenarery(vid,row['fromTime'])=='STOPPAGE':
                stoppage+=1
    return [driving,stoppage]
                

    

In [ ]:
def wire_issues_history(vid):
    wire_issues=pd.read_sql("select * from wire_issues where vehicle_id={}  order by day desc limit 30".format(vid),con=engine)
    
    if len(wire_issues)>0:
        print(wire_issues)
        
        a=[wire_issues['wire_loose'].sum(),wire_issues['ign_to_charge'].sum(),wire_issues['charge_to_ign'].sum(),wire_issues['intentional_wire_cut'].sum(),wire_issues['maintainence_wire_cut'].sum(),wire_issues['ign_wire_cut'].sum()]
    else:
        a=[None,None,None,None,None,None]
    return a

    
    

In [ ]:
def wire_issue(vid,fromTime,days):
    try:
        
        issues=wire_issue_custom_ping([vid,fromTime-days*86400,fromTime])[['intentional_wire_cut', 'maintainence_wire_cut','wire_loose', 'charge_to_ign', 'ign_wire_cut', 'ign_to_charge']]
        #print(issues.values)
        if len(issues)>0:
        
            return issues.values
        else:
        
            return [None,None,None,None,None]
    except:
        return [None,None,None,None,None]
        

    
    
    

In [ ]:
def no_info_history(vid):
    """returns [alert_count,avg_duration,sum_of_duration,median_of_duration]"""
    device_history=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':tm.time()-30*86400},'toTime':{'$gte':0},'vehicleId':vid})))  
    if len(device_history)>0:
        
        device_history['duration']=device_history['toTime']-device_history['fromTime']
        avg_duration=device_history['duration'].mean()
        sum_of_duration=device_history['duration'].sum()
        median_of_duration=device_history['duration'].median()
        alert_count=len(device_history)
        a=[alert_count,avg_duration,sum_of_duration,median_of_duration]
    else:
        a=[0,0,0,0]
    alert_count=len(device_history)
    return a
    

In [ ]:
def feature_generation(l):
    print(l)
    try:
        
        vid,fromTime,toTime=l
        basics=pd.read_sql("select vehicle_id,device_model_id,sales_dept,operator_id,network,installation_subtype from vehicle_details where vehicle_id={}".format(vid),con=engine).values.tolist()[0]
#        print(basics)

        no_info_features=no_info_history(vid)
#        print(no_info_features)

        wire_issues_features=wire_issue(vid,fromTime,2).tolist()[0]
        print("this is the output"+ str(wire_issues_features))

        last_and_next_state_features=last_state_before_no_info_history(vid)
#        print(last_and_next_state_features)

        current_state=[last_state_itenarery(vid,fromTime)]
#        print(current_state)

        fromTime_slot=[time_slot(fromTime)]
#       print(fromTime_slot)

        location_in=loc_before_no_info(vid=vid,time=fromTime)
#        print(location_in)

        displacement_travelled=[displacement(vid,fromTime,toTime)]
#        print(displacement_travelled)
        displacement_history_=displacement_history(vid)

        complete_features=basics+no_info_features+wire_issues_features+last_and_next_state_features+fromTime_slot+location_in+displacement_travelled+current_state+displacement_history_


        return complete_features
    except Exception as e:
        print('the issue with feature_generation is '+ str(e))
        return None
        
    
    
    
    
    
    
    

In [ ]:
device_history=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':tm.time()-6000},'toTime':{'$gte':0}})))    


In [ ]:
len(device_history)

In [ ]:
import multiprocessing
pool = multiprocessing.Pool()
results1=pool.map(feature_generation,device_history[['vehicleId','fromTime','toTime']].values.tolist()[0:5])
pool.close()
pool.join() 



In [ ]:
feature_names=['vehicle_id','device_model_id','sales_dept','operator_id','network','installation_subtype','alert_count','avg_duration','sum_of_duration','median_of_duration','intentional_wire_cut', 'maintainence_wire_cut','wire_loose', 'charge_to_ign', 'ign_wire_cut', 'ign_to_charge','driving_count','stoppage_count','time_slot','lat','long','displacement_travelled','current_state','displacements_mean','displacements_median','displacements_maximum','displacements_minimum']    



In [ ]:
complete_data=pd.DataFrame(results1,columns=feature_names)

In [ ]:
complete_data.to_csv("features_generated_data_4Nov.csv")

In [ ]:
complete_data


In [ ]:
complete_data=pd.read_csv("features_generated_data.csv")

In [ ]:
del complete_data['Unnamed: 0']

In [ ]:
complete_data

In [ ]:
data_notna=complete_data.dropna()

In [ ]:
data_notna

In [ ]:
import category_encoders as ce
import pygeohash as pgh

In [ ]:
data_notna.columns

In [ ]:
encoder_ordinal = ce.OrdinalEncoder(cols=['sales_dept','network','installation_subtype','intentional_wire_cut','maintainence_wire_cut','wire_loose','charge_to_ign', 'ign_wire_cut','ign_to_charge','current_state'])      
df_binary = encoder_ordinal.fit_transform(data_notna)


In [ ]:
df_binary['displacement_bucket']=df_binary['displacement_travelled'].apply(lambda x : 0 if x>1000 else 1)

In [ ]:
df_binary['loc']=df_binary.round(3)[['lat','long']].values.tolist()

In [ ]:
df_binary['encoded_loc']=df_binary['loc'].apply(lambda row: pgh.encode(row[0], row[1]))

In [ ]:
encoder_binary = ce.BinaryEncoder(cols=['encoded_loc'])
df_binary = encoder_binary.fit_transform(df_binary)


In [ ]:
df_binary['loc']

In [ ]:
features=df_binary[[ 'device_model_id', 'sales_dept', 'network',
       'installation_subtype', 'alert_count', 'avg_duration',
       'sum_of_duration', 'median_of_duration', 'intentional_wire_cut',
       'maintainence_wire_cut', 'wire_loose', 'charge_to_ign', 'ign_wire_cut',
       'ign_to_charge', 'driving_count', 'stoppage_count', 'time_slot', 'displacement_travelled', 'current_state', 'displacements_mean',
       'displacements_median', 'displacements_maximum',
       'displacements_minimum', 'encoded_loc_0', 'encoded_loc_1',
       'encoded_loc_2', 'encoded_loc_3']].values

In [ ]:
target=df_binary['displacement_bucket'].values

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

In [ ]:
model=LogisticRegression()

In [ ]:
model.fit(features,target)

In [ ]:
predictions=model.predict(features)

In [ ]:
from sklearn import metrics
from sklearn import preprocessing
from sklearn.model_selection import train_test_split



In [ ]:
sklearn?

In [ ]:
metrics.accuracy_score(target,predictions)

In [ ]:
features_train,features_test,target_train,taget_test=train_test_split(features,target,test_size=0.2,random_state=0)    



In [ ]:
model=LogisticRegression()

In [ ]:
model.fit(features_train,target_train)

In [ ]:
predictons=model.predict(features_test)

In [ ]:
metrics.accuracy_score(taget_test,predictons)

In [ ]:
from sklearn.feature_selection import RFE

In [ ]:
log_reg=LogisticRegression()

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
np.sqrt(-cross_val_score(log_reg,features,target,cv=3,scoring='neg_mean_squared_error')).mean()    



In [ ]:
df_binary.columns

In [ ]:
feauture_names=df_binary[[ 'device_model_id', 'sales_dept', 'network',
       'installation_subtype', 'alert_count', 'avg_duration',
       'sum_of_duration', 'median_of_duration', 'intentional_wire_cut',
       'maintainence_wire_cut', 'wire_loose', 'charge_to_ign', 'ign_wire_cut',
       'ign_to_charge', 'driving_count', 'stoppage_count', 'time_slot', 'displacement_travelled', 'current_state', 'displacements_mean',
       'displacements_median', 'displacements_maximum',
       'displacements_minimum', 'encoded_loc_0', 'encoded_loc_1',
       'encoded_loc_2', 'encoded_loc_3']].columns.values

In [ ]:
model=LogisticRegression()

In [ ]:
rfe=RFE(model)

In [ ]:
rfe.fit(features,target)

In [ ]:
sorted(zip(map(lambda x: round(x,3),rfe.ranking_),feauture_names))

In [ ]:
from sklearn.linear_model import Random

In [ ]:
len(target)

In [ ]:
a=pd.read_sql("select * from vahan_data",con=engine)

In [ ]:
len(a.vehicle_number.unique())